# Notebook 08 — Tu primer modelo de Machine Learning 🤖

¡Llegaste al territorio del **Machine Learning**! En los notebooks anteriores aprendiste a **explorar y visualizar** datos. Hoy das un paso más: vas a entrenar un **modelo** que **aprende un patrón** de los datos y luego **predice valores nuevos**.

## ¿Qué es Machine Learning supervisado?

Es darle a un algoritmo **muchos ejemplos de la forma `(entrada, respuesta correcta)`** para que descubra **la relación** entre ambos. Una vez aprendida, puedes darle entradas nuevas y obtener predicciones.

```
    Datos históricos                     Modelo entrenado
    ────────────────                     ─────────────────
    [edad, clase]  →  [fare]    ⟹     [edad, clase]  →  [fare predicho]
```

### Dos grandes tipos de problemas supervisados

| Tipo | Lo que predice | Ejemplo |
|---|---|---|
| **Regresión** | Un **número continuo** | Predecir el precio del billete (`fare`) |
| **Clasificación** | Una **categoría** | Predecir si un pasajero sobrevivió (sí/no) |

Hoy hacemos **regresión lineal**, el modelo más sencillo y más usado.

## Objetivos de aprendizaje

1. Entender la diferencia entre **features** (`X`) y **target** (`y`).
2. Hacer un **train/test split** y por qué es indispensable.
3. Entrenar un modelo `LinearRegression` de **scikit-learn**.
4. Hacer **predicciones** y evaluarlas con **R²** y **RMSE**.
5. **Interpretar los coeficientes** del modelo aprendido.

---

## 1. Setup

Cargamos `titanic` y lo limpiamos como en NB06.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Same cleanup as NB06
df = sns.load_dataset("titanic")
df_clean = df.drop(columns=["deck"]).copy()
df_clean["age"] = df_clean["age"].fillna(df_clean["age"].median())
df_clean = df_clean.dropna(subset=["embarked"]).reset_index(drop=True)

print(f"df_clean: {df_clean.shape[0]} rows, {df_clean.shape[1]} columns")
df_clean.head(3)

---

## 2. La pregunta del modelo

Vamos a intentar **predecir el precio del billete (`fare`)** a partir de **otras columnas** del dataset.

> ¿Pueden la **edad**, la **clase**, el **número de hermanos** y el **número de padres/hijos** a bordo, **explicar** cuánto pagó alguien por su billete?

### Variables del modelo

| Nombre | Símbolo | Rol | Columnas |
|---|---|---|---|
| **Features** (entradas) | `X` | Lo que el modelo **mira** | `age`, `pclass`, `sibsp`, `parch` |
| **Target** (salida) | `y` | Lo que el modelo debe **predecir** | `fare` |

> 💡 La convención es **`X` mayúscula** (porque es una **matriz** — varias columnas) y **`y` minúscula** (porque es un **vector** — una sola columna). Esta convención viene del álgebra lineal y es estándar en todo scikit-learn.

### 🏋️ Ejercicio 1 — Definir features y target

1. Crea un `DataFrame` llamado **`X`** con las columnas `["age", "pclass", "sibsp", "parch"]` de `df_clean`, en ese orden.
2. Crea una `Series` llamada **`y`** con la columna `"fare"` de `df_clean`.

In [ ]:
# YOUR CODE HERE
X = None
y = None


In [ ]:
# Tests
assert isinstance(X, pd.DataFrame), "X must be a pandas DataFrame"
assert isinstance(y, pd.Series), "y must be a pandas Series"

assert list(X.columns) == ["age", "pclass", "sibsp", "parch"], \
    f"X columns must be ['age','pclass','sibsp','parch'] in that order, got {list(X.columns)}"
assert X.shape == (889, 4), f"X shape must be (889, 4), got {X.shape}"
assert y.shape == (889,), f"y shape must be (889,), got {y.shape}"
assert y.name == "fare", f"y must come from the 'fare' column, got name '{y.name}'"

# No NaN allowed (since we cleaned df_clean)
assert X.isna().sum().sum() == 0, "X should not contain NaN"
assert y.isna().sum() == 0, "y should not contain NaN"

print("✅ ¡Bien! Definiste tus features (X) y tu target (y).")
print(f"   X: {X.shape}   y: {y.shape}")

---

## 3. Train/test split — la regla número uno del ML

Si entrenamos un modelo con **todos** los datos y luego medimos cuánto acierta sobre **esos mismos datos**, vamos a engañarnos. Es como estudiar para un examen con las preguntas exactas que vas a tener: tu nota no refleja lo que realmente sabes.

La solución: **dividir los datos en dos** *antes* de entrenar.

```
    Datos completos
    ──────────────────
            │
    ┌───────┴───────┐
    │               │
   80%             20%
   train           test
    │               │
   entrenar         evaluar
   el modelo        (datos que NUNCA vio)
```

### Sintaxis con scikit-learn

```python
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,        # 20% para test, 80% para train
    random_state=42,      # semilla — para que el split sea reproducible
)
```

> 💡 **`random_state=42`**: la división es aleatoria, pero fijar `random_state` hace que siempre obtengas la misma división. Esto es **importantísimo** para que tus resultados sean reproducibles.

### 🏋️ Ejercicio 2 — Train/test split

Importa `train_test_split` y crea las cuatro variables siguientes:

- **`X_train`**, **`X_test`**, **`y_train`**, **`y_test`**

Usa `test_size=0.2` y `random_state=42`.

In [ ]:
from sklearn.model_selection import train_test_split

# YOUR CODE HERE
X_train = None
X_test = None
y_train = None
y_test = None


In [ ]:
# Tests
assert X_train.shape == (711, 4), f"X_train shape must be (711, 4), got {X_train.shape}"
assert X_test.shape == (178, 4), f"X_test shape must be (178, 4), got {X_test.shape}"
assert y_train.shape == (711,), f"y_train shape must be (711,), got {y_train.shape}"
assert y_test.shape == (178,), f"y_test shape must be (178,), got {y_test.shape}"

# Sanity check on random_state (specific value the deterministic split should produce)
assert X_train.index[0] == 707, \
    "Train split looks different — did you set random_state=42?"

# Train and test indices must be disjoint
overlap = set(X_train.index) & set(X_test.index)
assert len(overlap) == 0, f"Train and test sets must NOT overlap (found {len(overlap)} shared rows)"

print("✅ ¡Bien! Tus datos están listos:")
print(f"   Train: {X_train.shape[0]} filas")
print(f"   Test:  {X_test.shape[0]} filas")

---

## 4. Entrenar la regresión lineal

Una **regresión lineal** asume que el target se puede aproximar como una **combinación lineal** de las features:

```
    fare ≈ β₀ + β₁·age + β₂·pclass + β₃·sibsp + β₄·parch
```

- Los **`βᵢ`** son los **coeficientes** que el modelo **aprende**.
- **`β₀`** es el **intercepto** (el valor de fare cuando todas las features son 0).
- El modelo busca los `βᵢ` que **minimizan el error** entre las predicciones y los valores reales en `y_train`.

### Sintaxis de scikit-learn (válida para CUALQUIER modelo)

```python
from sklearn.linear_model import LinearRegression

model = LinearRegression()      # 1. crear el modelo (vacío)
model.fit(X_train, y_train)     # 2. entrenarlo (aprende los coeficientes)
predictions = model.predict(X_test)   # 3. predecir
```

> 💡 **Esta misma estructura `fit / predict`** la vas a usar para **todos** los modelos de scikit-learn (KNN, Random Forest, redes neuronales con keras...). Aprenderla bien hoy te sirve para todo el bootcamp.

### 🏋️ Ejercicio 3 — Entrenar el modelo

1. Importa `LinearRegression` desde `sklearn.linear_model`.
2. Crea un objeto **`model`** de tipo `LinearRegression()`.
3. Entrénalo con `X_train` y `y_train` usando `.fit()`.

In [ ]:
from sklearn.linear_model import LinearRegression

# YOUR CODE HERE
model = None

# Then call .fit(...) on it


In [ ]:
# Tests
assert isinstance(model, LinearRegression), \
    f"model must be a LinearRegression instance, got {type(model).__name__}"

# After .fit(), the model has learned coef_ and intercept_
assert hasattr(model, "coef_"), "model is not fitted yet — did you call .fit(X_train, y_train)?"
assert hasattr(model, "intercept_"), "model is not fitted yet — did you call .fit(X_train, y_train)?"

assert model.coef_.shape == (4,), \
    f"Expected 4 coefficients (one per feature), got {model.coef_.shape}"

# pclass should have a strong NEGATIVE coefficient (3rd-class tickets are cheaper)
pclass_idx = list(X.columns).index("pclass")
assert model.coef_[pclass_idx] < -10, \
    f"pclass coefficient should be strongly negative (3rd class is cheaper), got {model.coef_[pclass_idx]:.2f}"

print("✅ ¡Genial! Tu modelo está entrenado.")
print(f"   Intercepto (β₀): {model.intercept_:.2f}")
for feature, coef in zip(X.columns, model.coef_):
    print(f"   Coeficiente de {feature}: {coef:+.2f}")

---

## 5. Predecir y evaluar — ¿qué tan bueno es el modelo?

Ahora que el modelo está entrenado, lo enfrentamos a los datos del **test** (que **nunca** vio durante el entrenamiento) y comparamos sus predicciones con los valores reales `y_test`.

### Métricas para regresión

| Métrica | Qué mide | Interpretación |
|---|---|---|
| **R²** (`r2_score`) | Fracción de la varianza explicada | 1 = perfecto, 0 = igual de bueno que predecir la media, <0 = peor que la media |
| **RMSE** (raíz del error cuadrático medio) | Error promedio en las **mismas unidades** del target | RMSE = 50 ⟹ el modelo se equivoca por unas 50 unidades en promedio |

```python
from sklearn.metrics import r2_score, mean_squared_error

predictions = model.predict(X_test)
r2 = r2_score(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
```

### 🏋️ Ejercicio 4 — Predecir y evaluar

1. Importa `r2_score` y `mean_squared_error` de `sklearn.metrics`.
2. Crea **`predictions`** llamando a `model.predict(X_test)`.
3. Crea **`r2`** = R² entre `y_test` y `predictions`.
4. Crea **`rmse`** = raíz cuadrada del MSE entre `y_test` y `predictions` (usa `np.sqrt`).

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error

# YOUR CODE HERE
predictions = None
r2 = None
rmse = None


In [ ]:
# Tests
assert predictions is not None, "predictions cannot be None"
assert hasattr(predictions, "shape"), "predictions should be a numpy array"
assert predictions.shape == (178,), f"predictions shape must be (178,), got {predictions.shape}"

assert isinstance(r2, float), f"r2 must be a float, got {type(r2).__name__}"
assert 0.25 < r2 < 0.45, \
    f"R² should be roughly between 0.25 and 0.45 with these features, got {r2:.3f}"

assert isinstance(rmse, float), f"rmse must be a float, got {type(rmse).__name__}"
assert 35 < rmse < 50, \
    f"RMSE should be roughly between 35 and 50, got {rmse:.2f}"

# Sanity: rmse must equal sqrt(mse)
expected_rmse = float(np.sqrt(mean_squared_error(y_test, predictions)))
assert np.isclose(rmse, expected_rmse), \
    f"rmse ({rmse:.4f}) doesn't match sqrt(mean_squared_error) ({expected_rmse:.4f})"

print("✅ ¡Excelente! Tu modelo predice con esta calidad:")
print(f"   R²   = {r2:.3f}    ({r2*100:.1f}% de la varianza explicada)")
print(f"   RMSE = {rmse:.2f}   (error promedio en las unidades de fare)")

---

## 6. Interpretar los coeficientes

Una de las cosas más bonitas de la regresión lineal es que los **coeficientes son interpretables**.

```
    coef de pclass = -34.2
```

> *"Por cada unidad que aumenta `pclass` (es decir, al pasar de 1ª a 2ª clase, o de 2ª a 3ª), el `fare` predicho **baja en ~34 unidades**, manteniendo todo lo demás constante."*

Y como `pclass` va de 1 (más cara) a 3 (más barata), un coeficiente negativo grande **tiene mucho sentido**.

### 🏋️ Ejercicio 5 — Coeficientes en una `Series` legible

Crea una **`Series`** llamada **`coefficients`**:

- Los **valores** son los coeficientes aprendidos (`model.coef_`).
- El **índice** son los nombres de las features (`X.columns`).

💡 Tip: `pd.Series(values, index=...)`.

In [ ]:
# YOUR CODE HERE
coefficients = None


In [ ]:
# Tests
assert isinstance(coefficients, pd.Series), \
    f"coefficients must be a pandas Series, got {type(coefficients).__name__}"
assert len(coefficients) == 4, f"Expected 4 coefficients, got {len(coefficients)}"
assert list(coefficients.index) == ["age", "pclass", "sibsp", "parch"], \
    f"Index must match feature names ['age','pclass','sibsp','parch'], got {list(coefficients.index)}"

# pclass coefficient must be strongly negative
assert coefficients["pclass"] < -10, \
    f"pclass coefficient should be strongly negative, got {coefficients['pclass']:.2f}"

# Coefficients must equal model.coef_ in the same order
assert np.allclose(coefficients.values, model.coef_), \
    "Coefficient values must match model.coef_ in the order of X.columns"

print("✅ ¡Bien! Coeficientes aprendidos por el modelo:")
print(coefficients.sort_values())
print(f"\nIntercepto: {model.intercept_:.2f}")

---

## 7. Visualización — predicciones vs. realidad

El último paso de cualquier evaluación de regresión: **graficar las predicciones contra los valores reales**. Si el modelo fuera perfecto, todos los puntos caerían sobre la diagonal `y = x`.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(y_test, predictions, alpha=0.5)

# Diagonal y = x — perfect predictions would land on this line
lims = [min(y_test.min(), predictions.min()), max(y_test.max(), predictions.max())]
ax.plot(lims, lims, color="red", linestyle="--", label="Predicción perfecta")

ax.set_xlabel("Fare real (y_test)")
ax.set_ylabel("Fare predicho")
ax.set_title(f"Predicciones vs. realidad — R² = {r2:.3f}")
ax.legend()
plt.show()

👀 ¿Qué ves?

- **Para fares bajos y medios** (la mayoría), el modelo se acerca razonablemente.
- **Para fares altos** (>100), el modelo **subestima sistemáticamente**: predice valores mucho más bajos que los reales.

Esto significa que con solo 4 features lineales **no se puede capturar toda la variabilidad** del precio. En el bootcamp aprenderás cómo mejorar esto con:

- **Más features** (especialmente categóricas como `embarked`, `sex`).
- **Modelos más expresivos** (Random Forest, Gradient Boosting).
- **Transformaciones de variables** (logaritmo de `fare`, por ejemplo).

---

## 8. Resumen — ¿qué aprendiste?

🎉 ¡Acabas de entrenar tu **primer modelo de Machine Learning**! Lo que viste hoy es el **flujo estándar** de cualquier proyecto supervisado en scikit-learn:

```
  ┌─────────────────────────────────────────────────────────────────────┐
  │  1. Definir X (features) e y (target)                               │
  │  2. train_test_split(X, y, test_size=..., random_state=...)         │
  │  3. model = LinearRegression()                                      │
  │  4. model.fit(X_train, y_train)                                     │
  │  5. predictions = model.predict(X_test)                             │
  │  6. evaluar con r2_score(y_test, predictions) o RMSE                │
  │  7. interpretar coeficientes / visualizar errores                   │
  └─────────────────────────────────────────────────────────────────────┘
```

### Conceptos clave

| Concepto | Idea |
|---|---|
| **Features (`X`) vs Target (`y`)** | Entradas vs. salida que queremos predecir |
| **Train / Test split** | Entrenar con unos datos, evaluar con otros |
| **`fit` / `predict`** | API universal de scikit-learn |
| **R²** | % de varianza explicada (más alto = mejor) |
| **RMSE** | Error promedio en las unidades del target |
| **Coeficientes** | Cuánto cambia el target por unidad de cada feature |

### Reglas prácticas

1. **Nunca** evalúes un modelo sobre los datos con los que lo entrenaste.
2. Fija `random_state` para reproducibilidad.
3. Empieza siempre con un modelo **simple** (regresión lineal) — te da un baseline contra el que comparar modelos más complejos.
4. Mira los **coeficientes** para entender qué aprendió el modelo (¿tiene sentido la dirección de cada uno?).
5. Grafica las predicciones contra los valores reales — los gráficos revelan errores que las métricas esconden.